# Testing the PPT² conjecture

The PPT² conjecture states that the composition of two PPT maps is entanglement
breaking. Equivalently, for any PPT state ρ the composite
τ = (I ⊗ Φ_ρ)(ρ) — built here by `ampliation(ρ, ρ, n, m)` — should be separable.
A counterexample is a PPT ρ whose composite τ is entangled.

This notebook samples random PPT states, composes each with itself, and tries to
certify entanglement of the composite by three independent criteria:

- **trace** — minimal value of ⟨form, τ⟩ over a library of pre-generated PNCP
  forms (a linear entanglement witness),
- **ampliation** — minimal eigenvalue of (I⊗form)(τ) over the same forms,
- **dps** — the level-2 DPS robustness from `Ket`.

A negative trace/ampliation value, or a positive robustness, flags entanglement.
No counterexample has been found: every value below stays on the separable side.

The detection criteria (`detect_trace`, `detect_ampliation`, `detect_dps`) and
the `test_ppt2` search driver live in the `ppt2` library (reached via
`using ppt2`, loaded by `common.jl`). The DPS criterion dominates the runtime —
expect a few seconds per trial in 4⊗4.

In [ ]:
include("common.jl")

## Load the precomputed PNCP forms

10,000 positive-but-not-completely-positive forms in 4⊗4, generated by
`scripts/gen_pncp.jl`. (No 3⊗3 form library is committed; for 3⊗3 use the
DPS criterion, which needs no forms, or generate forms first.)

In [ ]:
forms = load_batches("../pncp_4x4.jld2")
length(forms), size(forms[1])

## Run the conjecture test

Sample random PPT states and check each composite with `test_ppt2`, which runs
the criteria on one candidate and returns the evidence of the first criterion to
fire (or `nothing`). The loop here stops at the first detection.

In [ ]:
rng = Xoshiro(0)
state, evidence = nothing, nothing
@showprogress for _ in 1:100
    ρ = rand_ppt(4, 4; rng=rng)
    ev = test_ppt2(ρ; forms=forms)          # n=m=4, all criteria, compose=true by default
    if ev !== nothing
        global state, evidence = ρ, ev
        break
    end
end
(state = state, evidence = evidence)

### Integer-entry PPT states

The same search restricted to PPT states with entries in {-1, 0, 1} (cf. the old
`gen_ppt` notebook), obtained by passing a custom sampler to `rand_ppt`.

In [ ]:
int_sampler(rng) =
    rand_ppt(4, 4; rng=rng, rand_vec=(dims...; rng) -> float(rand(rng, -1:1, dims...)))

rng = Xoshiro(0)
hit = nothing
for _ in 1:100
    ev = test_ppt2(int_sampler(rng); criteria=(:trace, :ampliation), forms=forms)
    if ev !== nothing
        global hit = ev
        break
    end
end
hit

## Trace vs. ampliation sensitivity

(Salvaged from the old `ampliation_v_trace` notebook.) Both criteria use the same
form library, but the ampliation eigenvalue is the stronger test: a form that
fails to detect entanglement by trace can still do so under ampliation. We record
both statistics over a batch of composites and confirm they stay non-negative
(no counterexample).

In [ ]:
rng = Xoshiro(0)
trace_min = Float64[]
ampl_min  = Float64[]
@showprogress for _ in 1:50
    ρ = ppt2.rand_ppt(4, 4; rng=rng)
    τ = Hermitian(ampliation(ρ, ρ, 4, 4))
    push!(trace_min, detect_trace(τ, forms).value)
    push!(ampl_min,  detect_ampliation(τ, forms, 4, 4).value)
end
(trace = extrema(trace_min), ampliation = extrema(ampl_min))